# Project 2.0: Baseline Data and Equal-Weighted Portfolio

This notebook is the starting point of Project 2: **Markowitz Portfolio Optimization and Efficient Frontier Analysis**.

The purpose of Project 2.0 is to build the baseline dataset and construct an equal-weighted portfolio using the same stock universe as the previous Fama-French Portfolio Analysis project. While Project 1 focused on explaining the return drivers of an equal-weighted portfolio through factor models, this project will later examine whether mean-variance optimization can improve the portfolio's risk-return profile.

In this baseline stage, the analysis will:

1. Import Python packages.
2. Define the stock tickers and sample period.
3. Download adjusted close prices.
4. Check the data structure and missing values.
5. Calculate daily stock returns.
6. Construct equal-weighted portfolio returns.
7. Calculate cumulative returns for the equal-weighted portfolio.
8. Save the baseline datasets for later analysis.

The stock universe includes:

AAPL, MSFT, NVDA, AMZN, META, GOOGL, TSLA, JPM, XOM, and UNH.

The sample period is from **2019-01-01 to 2024-12-31**.

## 1. Import Python Packages

This section imports the Python packages needed for data collection, data cleaning, return calculation, and basic visualization.

In [1]:
# Install yfinance if it is not already available in the environment
!pip -q install yfinance

# Import Python packages
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

# Set basic plot size
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True

print("Packages imported successfully.")

Packages imported successfully.


## 2. Define Tickers and Date Range

This section defines the stock universe and the sample period used in this project.

The portfolio uses the same ten U.S. stocks from the previous Fama-French Portfolio Analysis project. This makes the Markowitz optimization project directly comparable to the earlier equal-weighted factor analysis.

In [2]:
# Define the stock universe
tickers = [
    "AAPL", "MSFT", "NVDA", "AMZN", "META",
    "GOOGL", "TSLA", "JPM", "XOM", "UNH"
]

# Define the sample period
start_date = "2019-01-01"
end_date = "2024-12-31"

# Check the basic setup
print("Number of stocks:", len(tickers))
print("Tickers:", tickers)
print("Start date:", start_date)
print("End date:", end_date)

Number of stocks: 10
Tickers: ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'META', 'GOOGL', 'TSLA', 'JPM', 'XOM', 'UNH']
Start date: 2019-01-01
End date: 2024-12-31


## 3. Download Adjusted Close Prices

This section downloads adjusted close price data from Yahoo Finance using `yfinance`.

Adjusted close prices are used because they account for corporate actions such as stock splits and dividends. This makes the price data more appropriate for return calculation.

In [3]:
# yfinance treats the end date as exclusive, so add one extra day for downloading
download_end_date = (pd.to_datetime(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

# Download stock price data from Yahoo Finance
stock_data = yf.download(
    tickers,
    start=start_date,
    end=download_end_date,
    auto_adjust=False,
    progress=False
)

# Extract adjusted close prices
adj_close_prices = stock_data["Adj Close"].copy()

# Keep columns in the same order as the ticker list
adj_close_prices = adj_close_prices[tickers]

print("Adjusted close price data downloaded successfully.")
print("Download end date used for yfinance:", download_end_date)

adj_close_prices.head()

Adjusted close price data downloaded successfully.
Download end date used for yfinance: 2025-01-01


Ticker,AAPL,MSFT,NVDA,AMZN,META,GOOGL,TSLA,JPM,XOM,UNH
Date,,,,,,,,,,
2019-01-02,37.503727,94.397156,3.376984,76.956497,134.623550,52.301727,20.674667,80.836510,50.001842,215.453186
2019-01-03,33.768066,90.924477,3.172957,75.014000,130.714249,50.853199,20.024000,79.687683,49.234135,209.577698
2019-01-04,35.209621,95.153305,3.376240,78.769501,136.875854,53.461643,21.179333,82.625420,51.049385,212.028748
2019-01-07,35.131248,95.274651,3.554981,81.475502,136.975098,53.355022,22.330667,82.682838,51.314846,212.435791
2019-01-08,35.800964,95.965454,3.466478,82.829002,141.420197,53.823650,22.356667,82.526909,51.687958,215.276199


## 4. Check Data Shape and Missing Values

Before calculating returns, it is important to check the structure and quality of the adjusted close price data.

This section checks the number of rows and columns, confirms the date range, and identifies whether any stocks contain missing price values.

In [4]:
# Check the shape of the adjusted close price data
print("Adjusted close price data shape:")
print(adj_close_prices.shape)

print("\nFirst available date:")
print(adj_close_prices.index.min())

print("\nLast available date:")
print(adj_close_prices.index.max())

print("\nMissing values by stock:")
print(adj_close_prices.isna().sum())

print("\nBasic information:")
adj_close_prices.info()

Adjusted close price data shape:
(1510, 10)

First available date:
2019-01-02 00:00:00

Last available date:
2024-12-31 00:00:00

Missing values by stock:
Ticker
AAPL     0
MSFT     0
NVDA     0
AMZN     0
META     0
GOOGL    0
TSLA     0
JPM      0
XOM      0
UNH      0
dtype: int64

Basic information:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1510 entries, 2019-01-02 to 2024-12-31
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    1510 non-null   float64
 1   MSFT    1510 non-null   float64
 2   NVDA    1510 non-null   float64
 3   AMZN    1510 non-null   float64
 4   META    1510 non-null   float64
 5   GOOGL   1510 non-null   float64
 6   TSLA    1510 non-null   float64
 7   JPM     1510 non-null   float64
 8   XOM     1510 non-null   float64
 9   UNH     1510 non-null   float64
dtypes: float64(10)
memory usage: 129.8 KB


## 5. Calculate Daily Stock Returns

This section calculates daily simple returns for each stock based on adjusted close prices.

Daily returns are calculated using percentage change:

Return = Current Price / Previous Price - 1

The first row is removed because returns cannot be calculated for the first available price observation.

In [5]:
# Calculate daily simple returns for each stock
daily_returns = adj_close_prices.pct_change().dropna()

# Display the first few rows
print("Daily returns calculated successfully.")
print("Daily returns shape:")
print(daily_returns.shape)

daily_returns.head()

Daily returns calculated successfully.
Daily returns shape:
(1509, 10)


Ticker,AAPL,MSFT,NVDA,AMZN,META,GOOGL,TSLA,JPM,XOM,UNH
Date,,,,,,,,,,
2019-01-03,-0.099608,-0.036788,-0.060417,-0.025241,-0.029039,-0.027696,-0.031472,-0.014212,-0.015354,-0.027270
2019-01-04,0.042690,0.046509,0.064067,0.050064,0.047138,0.051294,0.057697,0.036866,0.036870,0.011695
2019-01-07,-0.002226,0.001275,0.052941,0.034353,0.000725,-0.001994,0.054361,0.000695,0.005200,0.001920
2019-01-08,0.019063,0.007251,-0.024896,0.016612,0.032452,0.008783,0.001164,-0.001886,0.007271,0.013371
2019-01-09,0.016982,0.014300,0.019667,0.001714,0.011927,-0.003427,0.009483,-0.001690,0.005274,0.001438


## 6. Construct Equal-Weighted Portfolio Returns

This section constructs the baseline equal-weighted portfolio.

Since the portfolio contains ten stocks, each stock receives the same weight of 10%. The daily portfolio return is calculated as the average of the ten individual stock returns for each trading day.

This equal-weighted portfolio will be used later as the benchmark for comparing Markowitz optimized portfolios.

In [7]:
# Create equal weights for all stocks
equal_weights = np.repeat(1 / len(tickers), len(tickers))

# Calculate equal-weighted portfolio daily returns
equal_weighted_returns = daily_returns.dot(equal_weights)

# Convert the result to a DataFrame
equal_weighted_portfolio_returns = pd.DataFrame(
    equal_weighted_returns,
    columns=["Equal_Weighted_Return"]
)

# Display basic information
print("Equal-weighted portfolio returns calculated successfully.")
print("Number of stocks:", len(tickers))
print("Weight per stock:", equal_weights[0])
print("Equal-weighted portfolio returns shape:")
print(equal_weighted_portfolio_returns.shape)

equal_weighted_portfolio_returns.head()

Equal-weighted portfolio returns calculated successfully.
Number of stocks: 10
Weight per stock: 0.1
Equal-weighted portfolio returns shape:
(1509, 1)


,Equal_Weighted_Return
Date,
2019-01-03,-0.036710
2019-01-04,0.044489
2019-01-07,0.014725
2019-01-08,0.007919
2019-01-09,0.007567


## 7. Calculate Cumulative Returns

This section calculates the cumulative return of the equal-weighted portfolio.

Cumulative return shows how the portfolio grows over time if returns are continuously compounded through reinvestment. This baseline cumulative return will be used later to compare against optimized portfolios.

In [8]:
# Calculate cumulative return for the equal-weighted portfolio
# 把每日收益率连接起来，得到从 2019 到 2024 的整体累计表现
# 逻辑是:如果第一天涨 2%，第二天涨 3%，不是简单相加 5%，而是: (1 + 0.02) × (1 + 0.03) - 1
equal_weighted_portfolio_returns["Equal_Weighted_Cumulative_Return"] = (
    1 + equal_weighted_portfolio_returns["Equal_Weighted_Return"]
).cumprod() - 1

# Display basic information
print("Cumulative returns calculated successfully.")
print("Equal-weighted portfolio returns shape:")
print(equal_weighted_portfolio_returns.shape)

print("\nFirst five rows:")
display(equal_weighted_portfolio_returns.head())

print("\nLast five rows:")
display(equal_weighted_portfolio_returns.tail())

print("\nFinal cumulative return:")
print(equal_weighted_portfolio_returns["Equal_Weighted_Cumulative_Return"].iloc[-1])

Cumulative returns calculated successfully.
Equal-weighted portfolio returns shape:
(1509, 2)

First five rows:


,Equal_Weighted_Return,Equal_Weighted_Cumulative_Return
Date,,
2019-01-03,-0.036710,-0.036710
2019-01-04,0.044489,0.006146
2019-01-07,0.014725,0.020962
2019-01-08,0.007919,0.029046
2019-01-09,0.007567,0.036833



Last five rows:


,Equal_Weighted_Return,Equal_Weighted_Cumulative_Return
Date,,
2024-12-24,0.015377,5.929048
2024-12-26,-0.002362,5.912681
2024-12-27,-0.014628,5.811564
2024-12-30,-0.010786,5.738093
2024-12-31,-0.008422,5.681342



Final cumulative return:
5.681342215681439


## 8. Save and Download Baseline Datasets

This section saves the baseline datasets created in Project 2.0 and downloads them from Google Colab.

These files should be manually placed into the local `data/` folder of the GitHub repository after downloading.

In [12]:
# Save baseline datasets in the current Colab working directory
raw_stock_prices_file = "raw_stock_prices.csv"
daily_returns_file = "daily_returns.csv"
equal_weighted_returns_file = "equal_weighted_portfolio_returns.csv"

# Save datasets as CSV files
adj_close_prices.to_csv(raw_stock_prices_file)
daily_returns.to_csv(daily_returns_file)
equal_weighted_portfolio_returns.to_csv(equal_weighted_returns_file)

print("Baseline datasets saved successfully in the Colab working directory.")
print("Files created:")
print(raw_stock_prices_file)
print(daily_returns_file)
print(equal_weighted_returns_file)

from google.colab import files

files.download(raw_stock_prices_file)
files.download(daily_returns_file)
files.download(equal_weighted_returns_file)

Baseline datasets saved successfully in the Colab working directory.
Files created:
raw_stock_prices.csv
daily_returns.csv
equal_weighted_portfolio_returns.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>